# Imagen del dinosaurio (text-to-image)

Genera la imagen de Aborosaurus con `amused/amused-512` a partir del nombre y los rasgos que definió Ollama.
Ejecutar en Google Colab con GPU: **Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU (T4)**.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q diffusers transformers accelerate safetensors

In [ ]:
import json, torch
from diffusers import AmusedPipeline

MODELO = "amused/amused-512"
pipe = AmusedPipeline.from_pretrained(MODELO, variant="fp16", torch_dtype=torch.float16).to("cuda")

## Prompt

Sale de `ollama/identidad.json`. Si subes ese archivo a Colab se lee de ahí; si no, se usan los valores de abajo.
El prompt es corto: nombre + `prompt_imagen` (rasgos visibles en inglés).

In [ ]:
NOMBRE = "Aborosaurus"
PROMPT_IMAGEN = ("A medium-sized, slender dinosaur with long limbs clinging to a massive tree branch "
                 "in a dense, humid forest canopy. Olive green scales, browsing on leaves. Soft, dappled sunlight.")
try:
    with open("identidad.json", encoding="utf-8") as f:
        ident = json.load(f)["identidad"]
    NOMBRE, PROMPT_IMAGEN = ident["nombre"], ident["prompt_imagen"]
    print("Leído de identidad.json")
except FileNotFoundError:
    print("Usando los valores del notebook")

PROMPT = f"{NOMBRE}, a dinosaur. {PROMPT_IMAGEN}"
NEGATIVO = "blurry, low quality, text, watermark, deformed, extra limbs, cartoon"
PASOS, GUIA = 12, 10.0
print(PROMPT)

## Candidatas

Se generan cuatro imágenes con semillas distintas y se elige la que mejor corresponde a la descripción
(trepando en un árbol, esbelto, verde oliva, bosque húmedo).

In [ ]:
import time
from PIL import Image

SEMILLAS = [0, 1, 2, 3]
imagenes = []
for s in SEMILLAS:
    t = time.time()
    img = pipe(PROMPT, negative_prompt=NEGATIVO, num_inference_steps=PASOS, guidance_scale=GUIA,
               generator=torch.Generator("cuda").manual_seed(s)).images[0]
    print(f"semilla {s}: {time.time() - t:.1f} s")
    imagenes.append(img)

mosaico = Image.new("RGB", (512 * len(imagenes), 512))
for i, img in enumerate(imagenes):
    mosaico.paste(img, (512 * i, 0))
mosaico.save("candidatas.png")
mosaico

In [ ]:
ELEGIDA = 0  # semilla de la imagen que mejor corresponde a la descripción

imagenes[SEMILLAS.index(ELEGIDA)].save(f"{NOMBRE.lower()}.png")
info = {"nombre": NOMBRE, "modelo": MODELO, "prompt": PROMPT, "negative_prompt": NEGATIVO,
        "pasos": PASOS, "guidance_scale": GUIA, "semilla": ELEGIDA, "resolucion": "512x512",
        "gpu": torch.cuda.get_device_name(0)}
with open("imagen_info.json", "w", encoding="utf-8") as f:
    json.dump(info, f, indent=2, ensure_ascii=False)

from google.colab import files
for archivo in [f"{NOMBRE.lower()}.png", "candidatas.png", "imagen_info.json"]:
    files.download(archivo)